# Notebook 04: The Ape Framework and Simple Storage

As your smart contracts grow in complexity, writing custom Python scripts to deploy and test them becomes tedious.

Enter the **Ape Framework**! Ape is a smart contract development framework (similar to Hardhat or Brownie) built specifically for Python and Vyper developers. It handles compilation, deployment, and testing automatically.

In this notebook, we will:
1. Install the Ape framework.
2. Set up a basic project structure directly in Colab.
3. Write a Simple Storage contract.
4. Write tests for our contract.

### 1. Installation
We need to install `eth-ape` and its vyper plugin.

In [ ]:
!pip install eth-ape ape-vyper
!ape plugins install vyper

### 2. Project Setup

An Ape project requires a specific directory structure. We will create a `contracts` folder for our `.vy` files and a `tests` folder for our testing scripts.

In [ ]:
!mkdir -p ape_project/contracts
!mkdir -p ape_project/tests

# Create an empty ape-config.yaml file to initialize the project
!touch ape_project/ape-config.yaml

### 3. Writing the Contract

We will use Jupyter's `%%writefile` magic command to write our Vyper code directly into the `contracts` folder we just created.

This is a classic `SimpleStorage` contract. It has one state variable, `num`, a function to `store` a new number, and a function to `retrieve` it.

In [ ]:
%%writefile ape_project/contracts/SimpleStorage.vy
# @version ^0.3.0

num: uint256

@external
def store(num: uint256):
    self.num = num

@external
def retrieve() -> uint256:
    return self.num


Let's ask Ape to compile our project. It will look inside the `contracts` folder and compile any `.vy` files it finds.

In [ ]:
# Compile the contract
!cd ape_project && ape compile

### 4. Testing the Contract

Testing is the most important part of smart contract development. Bugs on the blockchain can't be easily fixed, and they can cost millions of dollars.

Ape uses the popular `pytest` framework under the hood. First, we write a `conftest.py` file to set up our testing environment (like deploying the contract before tests run).

In [ ]:
%%writefile ape_project/tests/conftest.py
import pytest

@pytest.fixture
def deployer(accounts):
    # Ape automatically provides 10 funded test accounts
    return accounts[0]

@pytest.fixture
def contract(project, deployer):
    # Deploy the SimpleStorage contract
    return project.SimpleStorage.deploy(sender=deployer)


Now, we write the actual tests to ensure `retrieve` and `store` work as expected.

In [ ]:
%%writefile ape_project/tests/test_simple_storage.py
def test_retrieve(contract):
    # By default, uint256 variables are initialized to 0
    num = contract.retrieve.call()
    assert num == 0

def test_store(contract, deployer):
    # We must specify the 'sender' when modifying state
    contract.store(19, sender=deployer)
    
    # Check if it updated
    num = contract.retrieve.call()
    assert num == 19


Let's run the tests!

In [ ]:
!cd ape_project && ape test